In [5]:
import numpy as np
import netgen.csg as csg
import netgen.occ as occ
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw

In [ ]:
# Import and smooth Stanford Bunny
import sys 
sys.path.insert(0, "/Users/andrmas/Software/cosmos/sandbox")
print(sys.path)

In [ ]:
from generate_surface_meshes import *

In [ ]:
fname = "../../data/geometries/bunny.stl"
bunny_mesh, _ = import_stl_mesh(fname,maxh=4)
Draw(bunny_mesh)

In [ ]:
fname = "../../data/geometries/schwarz-p-single_fixed.stl"
schwarz_p_single_mesh, schwarz_geo = import_stl_mesh(fname,maxh=4)
schwarz_p_single_mesh.Curve(3)
Draw(schwarz_p_single_mesh)

In [ ]:
fname = "../../data/geometries/chwarz-p-single_fixed_open.stl"
schwarz_p_single_mesh, schwarz_geo = import_stl_mesh(fname,maxh=4)
schwarz_p_single_mesh.Curve(3)
Draw(schwarz_p_single_mesh)

In [ ]:
fname = "../../data/geometries/schwarz-p-single.stl"
schwarz_p_mesh = import_stl_mesh(fname,maxh=4)
Draw(schwarz_p_mesh)

In [11]:
fname = "../../data/geometries/genus_2_surface.stl"
geo = stl.STLGeometry(fname)
maxh = 0.01
ngmesh = geo.GenerateMesh(perfstepsend=MeshingStep.MESHSURFACE,
                          quad=False,
                          maxh=maxh,
                        #   grading=0.7,
                          optimize2d = "smsmsmSmSmSm",
                        #   yangle=50,
                        #   contyangle=50,
                        #   edgecornerangle=30
                          )
mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw(genus_2_mesh)

In [ ]:
# Define geometry related objects and differential operators

# Mesh size
h = specialcf.mesh_size

# Normal vector
ns = specialcf.normal(mesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs
# Weingarten map
Bs = Grad(ns)
# TODO: Or should we rather use
# specialcf.Weingarten(mesh.dim)

# Define covariant derivative for vectors
def Nabla_h(u):
    return Ps*Grad(u).Trace()*Ps
# and its corrected version for non-tangential vectors
def Nabla_th(u):
    return Nabla_(u) - InnerProduct(u, ns)*Bs

# Define discrete strain rate tensor
# TODO: Need to clarify meaning/implementation of Grad(u).Trace()
#       for VectorH1
# TODO: Which projection Ps do we actually need
def E_h(u):
    return 0.5*Ps*Sym(Grad(u).Trace())*Ps
    # return 0.5*Ps*(Grad(u).Trace())
    # return Ps*(grad(u).Trace())

# Corrected strain rate tensor for non-tangential 
def E_th(u):
    return E_h(u) - InnerProduct(u, ns)*Bs

In [ ]:
V = VectorH1(mesh, order=int(order_k))
Q = H1(mesh, order=int(order_k-1))
N = NumberSpace(mesh)
W = V*Q*N

# V = HDivSurface(mesh, order=int(order_k))
# V = H1(mesh, order=int(order_k))
(u, p, lam), (v, q, mu) = W.TnT()

# Penalty parameter to weakly enforce tangential condition
beta = 1.0

Astar = BilinearForm(W)
Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
# TODO: DO we need Traces here?
Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
# TODO: Last term can replaced with
# Astar += InnerProduct(u.Trace(), v.Trace())*ds
# Penalization of normal component
# TODO:More efficient way would be to compute (u,n)*(v*n) directly
# Test that!
Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
# Add b_h contributions 
Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
Astar += (p*mu + q*lam)*ds
Astar.Assemble()




In [ ]:
# Tangential test field
uex = CF((-y*(1-z)*(1+z), x*(1-z)*(1+z), 0))
# pex = sin(x)*sin(y)*sin(z)
pex = x*y+y*z+z*x

uex_h, pex_h, _ = GridFunction(W).components
uex_h.Set(uex, definedon=mesh.Boundaries(".*"))
pex_h.Set(pex, definedon=mesh.Boundaries(".*"))

l = LinearForm(W)
l += 2*InnerProduct(E_th(uex_h), E_th(v))*ds
l += InnerProduct(Ps*uex_h.Trace(), Ps*v.Trace())*ds
l += beta*h**(-2.0)*InnerProduct(Qs*uex_h.Trace(), Qs*v.Trace())*ds
l += InnerProduct(grad(pex_h).Trace(), v.Trace())*ds
l += InnerProduct(grad(q).Trace(), uex_h.Trace())*ds
l += (pex_h*mu)*ds

l.Assemble()

In [ ]:
# Solve problem and write out solution
up_h =  GridFunction(W)
Ainv = Astar.mat.Inverse(freedofs=W.FreeDofs())
up_h.vec.data = Ainv*l.vec

u_h, p_h, _ = up_h.components

Draw(uex_h)
Draw(u_h)
Draw(pex_h)
Draw(p_h)
vtkfile = VTKOutput(mesh, coefs=[u_h, uex_h, p_h, pex_h], names=["u_h", "uex_h", "p_h", "pex_h"], filename="results_stokes",subdivision=3)
vtkfile.Do(vb=BND)
